In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor    

In [2]:
data = pd.read_csv(r"D:\Data Science_bat\Machine Learning\Raw_datasets\insurance.csv")

cat_col = ['gender','region','smoker']
data_dummies = pd.get_dummies(data,columns = cat_col, drop_first = True)
X = data_dummies[['age','bmi','children','gender_male','region_northwest','region_southeast','region_southwest','smoker_yes']]
Y = data[['charges']]

# train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size = 0.20, random_state = 42)
test_df = pd.concat ([X_test,y_test],axis =1 )


In [3]:
data

,age,gender,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [4]:
model = XGBRegressor()
model.fit(X_train,y_train)
xgb_pred = model.predict(X_test)
xgb_pred = [round(value) for value in xgb_pred]
test_df['xgb_pred'] = xgb_pred
# test_df.to_csv('gbm_reg_results.csv')
mse = mean_squared_error(y_test, xgb_pred)
print(mse)

23434488.38066501


In [11]:
model = XGBRegressor(eta = 0.006, learning_rate=0.01)
model.fit(X_train,y_train)
xgb_pred = model.predict(X_test)
xgb_pred = [round(value) for value in xgb_pred]
test_df['xgb_pred'] = xgb_pred
# test_df.to_csv('gbm_reg_results.csv')
mse = mean_squared_error(y_test, xgb_pred)
print(mse)

37407205.05917153


In [18]:
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV,RandomizedSearchCV
from sklearn.metrics import mean_squared_error

# Model (keep your params; note eta and learning_rate are aliases—see note below)
model = XGBRegressor(
    eta=0.006,
    learning_rate=0.01,
    random_state=42
)


eta_values = [0.001, 0.002, 0.003, 0.004, 0.005,
              0.006, 0.007, 0.008, 0.009, 0.010]   # 10 values

learning_rate_values = [0.01, 0.02, 0.03, 0.04, 0.05,
                        0.06, 0.07, 0.08, 0.09, 0.10]  # 10 values

param_grid = {
    "eta": eta_values,
    "learning_rate": learning_rate_values
}

# CV splitter
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# # CV using negative MSE scoring
# cross_validator = cross_val_score(
#     model,
#     X_train,
#     y_train,
#     cv=kf,
#     n_jobs=1,
# )

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=kf,
    n_jobs=1,
    verbose=1,
    refit=True
)

grid.fit(X_train,y_train)


best_model = grid.best_estimator_
best_params = grid.best_params_
best_cv_mse = -grid.best_score_

print("Best params:", best_params)
print("Best CV MSE:", best_cv_mse)
print("Best CV RMSE:", np.sqrt(best_cv_mse))

gb_pred = best_model.predict(X_test)

# If your target is integer-like and you want rounding (same as your code):
xgb_pred_round = np.round(xgb_pred)

test_df["xgb_pred"] = xgb_pred_round

mse = mean_squared_error(y_test, xgb_pred_round)
print("Test MSE (rounded preds):", mse)

# Also print unrounded MSE (recommended for regression)
mse_unrounded = mean_squared_error(y_test, xgb_pred)
print("Test MSE (raw preds):", mse_unrounded)


Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params: {'eta': 0.001, 'learning_rate': 0.03}
Best CV MSE: 23978365.4
Best CV RMSE: 4896.770915613676
Test MSE (rounded preds): 37407205.05917153
Test MSE (raw preds): 37407205.05917153


In [14]:
{'precision_samples', 'recall_weighted', 'rand_score', 'roc_auc', 'roc_auc_ovr', 'accuracy', 'top_k_accuracy', 'adjusted_rand_score', 'f1', 'precision', 'f1_samples', 'normalized_mutual_info_score', 'v_measure_score', 'explained_variance', 'recall_macro', 'f1_macro', 'roc_auc_ovo_weighted', 'neg_mean_absolute_error', 'neg_log_loss', 'jaccard_micro', 'neg_mean_poisson_deviance', 'r2', 'roc_auc_ovr_weighted', 'balanced_accuracy', 'mutual_info_score', 'precision_macro', 'precision_micro', 'matthews_corrcoef', 'recall', 'jaccard_weighted', 'neg_mean_absolute_percentage_error', 'jaccard', 'average_precision', 'homogeneity_score', 'completeness_score', 'neg_root_mean_squared_log_error', 'neg_max_error', 'neg_brier_score', 'neg_mean_gamma_deviance', 'f1_micro', 'neg_negative_likelihood_ratio', 'neg_mean_squared_error', 'neg_median_absolute_error', 'jaccard_samples', 'recall_micro', 'neg_root_mean_squared_error', 'precision_weighted', 'positive_likelihood_ratio', 'adjusted_mutual_info_score', 'roc_auc_ovo', 'f1_weighted', 'neg_mean_squared_log_error', 'd2_absolute_error_score', 'jaccard_macro', 'fowlkes_mallows_score', 'recall_samples'}

{'accuracy',
 'adjusted_mutual_info_score',
 'adjusted_rand_score',
 'average_precision',
 'balanced_accuracy',
 'completeness_score',
 'd2_absolute_error_score',
 'explained_variance',
 'f1',
 'f1_macro',
 'f1_micro',
 'f1_samples',
 'f1_weighted',
 'fowlkes_mallows_score',
 'homogeneity_score',
 'jaccard',
 'jaccard_macro',
 'jaccard_micro',
 'jaccard_samples',
 'jaccard_weighted',
 'matthews_corrcoef',
 'mutual_info_score',
 'neg_brier_score',
 'neg_log_loss',
 'neg_max_error',
 'neg_mean_absolute_error',
 'neg_mean_absolute_percentage_error',
 'neg_mean_gamma_deviance',
 'neg_mean_poisson_deviance',
 'neg_mean_squared_error',
 'neg_mean_squared_log_error',
 'neg_median_absolute_error',
 'neg_negative_likelihood_ratio',
 'neg_root_mean_squared_error',
 'neg_root_mean_squared_log_error',
 'normalized_mutual_info_score',
 'positive_likelihood_ratio',
 'precision',
 'precision_macro',
 'precision_micro',
 'precision_samples',
 'precision_weighted',
 'r2',
 'rand_score',
 'recall',
 're